In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import random
from datetime import datetime as dt, timedelta

In [ ]:
file_id = '1HEDY7b-ifLt98_uum4-pJeAMay9hZzgV'
url = f'https://drive.google.com/uc?export=download&id={file_id}'
df = pd.read_csv(url)
def add_expense(records, current_date, product, budget=None, chance=False, isleak=False):
    """Add expense based on chance and budget constraints"""
    if random.random() < chance:
        hour = random.randint(0, 23)
        minute = random.randint(0, 59)

        # Extract product details
        product_name = product['product_clean'].values[0]
        master_category = product['Master_Category'].values[0]
        macro_category = product['Macro_Category'].values[0]

        # Pembulatan harga produk dari dataset ke ratusan terdekat
        price = round(product['price'].values[0] / 100) * 100

        if isleak:
            records.append([current_date.replace(hour=hour, minute=minute), 'Expense', price, macro_category, master_category, product_name])
            return budget # Budget tidak terpengaruh untuk logic isleak
        else:
            new_budget = budget - price
            # Batal beli jika budget tidak cukup
            if new_budget < 0:
                return budget
            else:
                records.append([current_date.replace(hour=hour, minute=minute), 'Expense', price, macro_category, master_category, product_name])
                return new_budget
    else:
        return budget

def generate_expenses(df, start_date, num_days=180):
    """Membuat mockup pengeluaran yang lebih realistis dan terstruktur"""

    # PERBAIKAN ERROR NaN: Buang baris data produk yang harganya kosong/NaN
    df = df.dropna(subset=['price']).copy()

    # Standarisasi huruf kategori agar ML tidak bingung (Needs vs NEEDS)
    if 'Macro_Category' in df.columns:
        df['Macro_Category'] = df['Macro_Category'].str.title()

    # Tentukan seed agar hasil tetap konsisten saat dievaluasi
    np.random.seed(70)
    random.seed(70)
    # Seed leak = 70
    # Seed Non-leak = 69

    records = []
    current_date = dt.strptime(start_date, '%Y-%m-%d')
    end_date = current_date + timedelta(days=num_days)

    # Determine fixed monthly income & expense
    salary = random.randint(4000000, 10000000)
    rent = random.randint(952000, 2240000)
    listrik = random.randint(200000, 448000)

    # Simpan BASE budget agar bisa di-reset tiap awal bulan
    base_kebutuhan_rumah = random.randint(280000, 784000)
    base_lain_lain = random.randint(168000, 560000)

    # Inisialisasi awal
    kebutuhan_rumah = base_kebutuhan_rumah
    lain_lain = base_lain_lain

    # Menghitung routine expenses harian dengan pembulatan ke ratusan terdekat
    daily_transport = round((random.randint(168000, 476000) / 20 / 2) / 100) * 100
    daily_meal = round((random.randint(1528000, 2699200) / 30 / 3) / 100) * 100

    leak = random.choice([True, False])
    kebutuhan_rumah_chance = random.random()
    lain_lain_chance = random.random()

    # OPTIMASI: Filter dataframe di luar loop agar iterasi harian lebih cepat
    df_wants = df[df['Macro_Category'] == 'Wants']
    df_kebutuhan_rumah = df[df['Master_Category'] == 'Kebutuhan Rumah & Mandi']
    df_lain_lain = df[df['Master_Category'] == 'Lain-lain & Darurat']
    leak_product = df_wants.sample(1)

    while current_date < end_date:
        # Inject salary & reset budget every month (at 1st of the month)
        if current_date.day == 1:
            records.append([current_date.replace(hour=9, minute=0), 'Income', salary, 'Needs', 'Salary', 'Gaji Bulanan'])
            records.append([current_date.replace(hour=10, minute=0), 'Expense', rent, 'Needs', 'Tagihan & Kewajiban', 'Bayar Kosan'])
            records.append([current_date.replace(hour=10, minute=15), 'Expense' ,  listrik, 'Needs', 'Tagihan & Kewajiban', 'Listrik'])

            # Reset budget variabel tiap pergantian bulan
            kebutuhan_rumah = base_kebutuhan_rumah
            lain_lain = base_lain_lain

        # Add routine expense (Transport hanya hari kerja Senin-Jumat)
        if current_date.weekday() < 5:
            records.append([current_date.replace(hour=8, minute=0), 'Expense' , daily_transport, 'Needs', 'Transportasi & Rutinitas', 'GoJek pergi'])
            records.append([current_date.replace(hour=17, minute=0), 'Expense' , daily_transport, 'Needs', 'Transportasi & Rutinitas', 'GoJek pulang'])
        # Add additional expense (Transport pada hari sabtu-minggu)
        elif current_date.weekday() >= 5 & random.choice([True, False]):
            records.append([current_date.replace(hour=8, minute=0), 'Expense' , daily_transport, 'Wants', 'Transportasi & Rutinitas', 'GoJek pergi'])
            records.append([current_date.replace(hour=17, minute=0), 'Expense' , daily_transport, 'Needs', 'Transportasi & Rutinitas', 'GoJek pulang'])


        # Makan rutin setiap hari
        records.append([current_date.replace(hour=7, minute=0), 'Expense', daily_meal, 'Needs', 'Makan & Minum Harian', 'Sarapan'])
        records.append([current_date.replace(hour=12, minute=0), 'Expense', daily_meal, 'Needs', 'Makan & Minum Harian', 'Makan Siang'])
        records.append([current_date.replace(hour=18, minute=0), 'Expense', daily_meal, 'Needs', 'Makan & Minum Harian', 'Makan Malam'])

        # Leak_constant logic
        if leak and not df_wants.empty:
            leak_chance = random.uniform(0.7, 1)
            add_expense(records, current_date, leak_product, None, leak_chance, True)

        # Leak_normal logic
        if not df_wants.empty:
          leak_product_varies = df_wants.sample(1)
          leak_chance = 0.2
          add_expense(records, current_date, leak_product_varies, None, leak_chance, True)

        # Add kebutuhan rumah expense
        if not df_kebutuhan_rumah.empty:
            product_kebutuhan_rumah = df_kebutuhan_rumah.sample(1)
            kebutuhan_rumah = add_expense(records, current_date, product_kebutuhan_rumah, kebutuhan_rumah, chance=kebutuhan_rumah_chance, isleak=False)

        # Add lain-lain expense
        if not df_lain_lain.empty:
            product_lain_lain = df_lain_lain.sample(1)
            lain_lain = add_expense(records, current_date, product_lain_lain, lain_lain, chance=lain_lain_chance, isleak=False)

        current_date += timedelta(days=1)

    # Build DataFrame
    df_final = pd.DataFrame(records, columns=['timestamp', 'type', 'amount', 'macro_category', 'master_category', 'title'])

    # Sort by timestamp to ensure chronological order & reset index
    df_final = df_final.sort_values(by='timestamp').reset_index(drop=True)

    return df_final # Ganti nilainya ke leak kalau mau tau data leak atau ngga yang di generate

# Memanggil fungsi
mock_finance_df = generate_expenses(df, '2025-01-01', num_days=180)
mock_finance_df